# PM02 — Canopy Pipeline Development Notebook
**CS 440 / CS 540 — Data Engineering**

**Team name:** Enter name
**Team members:** Enter names

## Purpose
Use this notebook to explore the input data, prototype PySpark operations, and document engineering decisions. Implement the actual pipeline in the required Python modules; the notebook is not a substitute for those modules.

Use PySpark on Databricks Serverless, not Pandas. Preserve meaningful experiments, results, and explanations for submission.

## 1. Environment and input access
Update the path for your team's Unity Catalog volume. This is a quick exploration read; production ingestion belongs in `ingest.py`.

**Expected input:** 50,000 records and 25 columns.

In [ ]:
INPUT_PATH = "/Volumes/workspace/default/de_course_data/canopy_project_data.csv"

raw_df = spark.read.option("header", True).csv(INPUT_PATH)
print(f"Records: {raw_df.count():,}")
print(f"Columns: {len(raw_df.columns)}")
raw_df.printSchema()
display(raw_df.limit(10))

**Setup notes:** Record your input path and any observations.

## 2. Data exploration
Use PySpark to investigate column types, nulls and blanks, exact duplicate records, suspicious numeric values, and date relationships.

Document at least three specific findings and explain how each influenced your implementation. Include supporting output from your experiments.

In [ ]:
# Your exploration code here.






### Findings

1. **Evidence:** ... **Engineering implication:** ...
2. **Evidence:** ... **Engineering implication:** ...
3. **Evidence:** ... **Engineering implication:** ...

## 3. Cleaning experiments
Prototype trimming, conversion of blank strings to null, and exact duplicate removal. Show how you checked that legitimate distinct records were preserved.

Transfer working logic to `clean.py`. Explain your decisions below.

In [ ]:
# Your cleaning experiments here.

col_name = "product_category"

clean = raw_df.withColumn(
    col_name,
    F.when(F.trim(F.col(col_name)) == "", F.lit(None)).otherwise(F.trim(F.col(col_name)))
)


print("Padded before:", raw_df.filter(F.col(col_name) != F.trim(F.col(col_name))).count())
print("Blank before: ", raw_df.filter(F.trim(F.col(col_name)) == "").count())
print("Nulls before: ", raw_df.filter(F.col(col_name).isNull()).count())

# After
print("Padded after: ", clean.filter(F.col(col_name) != F.trim(F.col(col_name))).count())
print("Blank after:  ", clean.filter(F.col(col_name) == "").count())
print("Nulls after:  ", clean.filter(F.col(col_name).isNull()).count())




**Cleaning decisions and evidence:** ...
1. Input: The data or conditions tested.
    - i checked on of the columns product_category for it having padding having a blank string or being null
2. Expected result: What the specification requires.
    - i expected to either find values or for the data to be clean
3. Actual result: The observed output.
    - the data was clean for that column
4. Conclusion: Whether the result was correct and what you learned
    - that the product_category was clean and does not need anything cleaning at the moment (Note code should still include it incase of future problems)


## 4. Validation experiments
Prototype the assignment's data-quality rules and record all applicable rejection reasons for invalid rows. Include a deliberately constructed row with multiple violations.

Follow the PM02 specification for conditional fields: a null `self_service_attempted` value alone is not a rejection reason.

Transfer working logic to `validate.py`.

In [ ]:
# Your validation experiments here.
col_name = "customer_id"

total = raw_df.count()
invalid = raw_df.filter(F.col(col_name).isNull()).count()
valid = raw_df.filter(F.col(col_name).isNotNull()).count()

print("Total rows:   ", total)
print("Invalid (null customer_id):", invalid)
print("Valid (has customer_id):   ", valid)
print("Invalid + valid = total:   ", invalid + valid == total)

1. Input: The data or conditions tested.
    - the costomer_id colunm was scanned for null values
2. Expected result: What the specification requires.
    - I expected some values to be null but most to be valid
3. Actual result: The observed output.
    - 501 invalid values with the reaming 49499 being good
4. Conclusion: Whether the result was correct and what you learned
    - That the customer_id colunm needs verification

## 5. Transformation experiments
Prototype the five required derived fields using PySpark:

* `gross_amount`
* `discount_amount`
* `net_amount`
* `delivery_delay_days`
* `supplier_receipt_delay_days`

Check representative numeric and date cases, including applicable null behavior. Transfer working logic to `transform.py`.

In [ ]:
# Your transformation experiments here.
from pyspark.sql import functions as F
raw_df = raw_df.withColumn(
    "gross_amount",
    F.round(
        F.col("quantity").cast("double") *
        F.col("unit_price").cast("double"),
        2
    )
).withColumn(
    "discount_amount",
    F.round(
        F.col("gross_amount") * F.col("discount").cast("double"),
        2
    )
).withColumn(
    "net_amount",
    F.round(
        F.col("gross_amount") - F.col("discount_amount"),
        2
    )
)

display(raw_df.select(
    "quantity", "unit_price", "discount",
    "gross_amount", "discount_amount", "net_amount"
))

**Transformation checks:** Document at least two expected-versus-actual results.
1. input: quantity, unit price, discount quantity:2 unit price: 44.76 discount: 0.1
2. Expected result: What the specification requires. Gross amount:89.52, discount Amount:8.95, and net amount:80.57
3. Actual result: The observed output. row 2 exact same daata as expected
4. conclusion: Whether the result was correct and what you learned result was the same, and learned some basic math to add discounts into calculations.

## 6. Engineering decisions
Answer concisely, with reference to your experiments.

1. What important data-quality problems did you discover?
2. Which cleaning or validation decision was most difficult, and what alternative did you consider?
3. How did you verify that the transformations were correct?
4. What would change if the pipeline processed 50 million records instead of 50,000?

**Your answers:**

1. ...
2. ...
3. ...
4. ...

> **Submission reminder:** Keep the notebook as evidence of development. The Python application must independently run the pipeline.

In [ ]:
# Optional final diagnostic experiment.